In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from utils import (
    load_config_json,
    scale_config_to_resolution,
    load_and_preprocess_image,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    detect_and_evaluate_ostia,
    visualize_aorta_with_ostia,
    segment_arteries_from_ostia,
    visualize_arteries_comparison,
)

# Análise de Casos Ruins

In [2]:
# Configurações
CONFIG_PATH = "../config/pipeline_config.json"
DATA_PATH = "/media/matheus/HD/DatasetsCCTA/ImageCAS/1-1000" # "/data04/home/mpmaia/ImageCAS/database/1-1000"
OUTPUT_PATH = "../output"
USE_CACHE = True
HTML_OUTPUT_DIR = Path(f"{OUTPUT_PATH}/cases_analysis_3d")
HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

# Carregar config padrão do arquivo com uma base config mínima
base_config = {}
CONFIG = load_config_json(CONFIG_PATH, base_config)

# Carregar dados de bad cases
bad_cases_high = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_high_res.csv")
bad_cases_low = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_mid_res.csv")

# Selecionar casos aleatórios
np.random.seed(RANDOM_SEED)
n_samples = 1

# Amostrar dos casos ruins
high_sample = bad_cases_high.sample(n=min(n_samples, len(bad_cases_high)), random_state=RANDOM_SEED)
low_sample = bad_cases_low.sample(n=min(n_samples, len(bad_cases_low)), random_state=RANDOM_SEED)

selected_cases = pd.concat([
    high_sample.assign(resolution='high'),
    low_sample.assign(resolution='mid')
], ignore_index=True)

print(f"\nSelecionados {len(selected_cases)} casos para análise")
print("\nCasos Selecionados:")
print(selected_cases[['image_id', 'bad_case_status', 'resolution']])


Selecionados 2 casos para análise

Casos Selecionados:
   image_id bad_case_status resolution
0       431           error       high
1       498     one_correct        mid


In [ ]:
def run_pipeline_for_case(img_id, config, data_path=None, save_base_path=None):
    """Executa o pipeline completo para uma imagem e retorna os resultados"""
    if data_path is None:
        data_path = str(DATA_PATH)
    print(f"\n{'='*60}")
    print(f"Processando IMG {img_id}")
    print(f"{'='*60}")

    try:
        resolution = 'high' if config['DOWNSCALE_FACTORS'] == [1, 1, 1] else 'mid'
        cache_root = save_base_path or f"{OUTPUT_PATH}/cases_analysis_cache"
        vesselness_cache_dir = f"{cache_root}/vesselness_{resolution}"
        pipeline_cache_dir = f"{cache_root}/pipeline_{resolution}"
        artery_cache_dir = f"{cache_root}/artery_{resolution}"

        run_config = dict(config)
        run_config['LOAD_CACHE'] = USE_CACHE
        run_config['SAVE_CACHE'] = USE_CACHE

        # 1. Carregar e pré-processar
        print("1. Carregando e pré-processando a imagem...")
        image_data = load_and_preprocess_image(img_id, data_path, run_config)
        lcc_image = image_data['lcc_image']
        label = image_data['label']
        downscale_factors = image_data['downscale_factors']
        scaled_spacing = image_data['scaled_spacing']

        del image_data # Liberar memória

        # 2. Detectar circulos aorta
        print("2. Detectando óstios (círculos na aorta)...")
        detected_circles = get_or_detect_aorta_circles(
            img_id, lcc_image, downscale_factors, scaled_spacing,
            run_config['CIRCLE_DETECTION'], base_save_path=pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 3. Segmentar aorta
        print("3. Segmentando aorta...")
        aorta_mask = get_or_segment_aorta(
            img_id, lcc_image, detected_circles,
            run_config['LEVEL_SET'], pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 4. Calcular vesselness para ostios
        print("4. Calculando vesselness para ostios...")
        vesselness_ostios = get_or_compute_vesselness(
            img_id, lcc_image, cache_dir=vesselness_cache_dir,
            vesselness_config=run_config['VESSELNESS_AORTA'],
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        # 5. Detectar e avaliar óstios
        print("5. Detectando e avaliando óstios...")
        try:
            ostia_results = detect_and_evaluate_ostia(
                aorta_mask, vesselness_ostios, label, scaled_spacing, run_config
            )
            ostia_left = ostia_results['ostia_left']
            ostia_right = ostia_results['ostia_right']
            label_artery = ostia_results['label_artery']
        except ValueError as ostia_exc:
            print(f"Erro na detecção/avaliação dos óstios: {ostia_exc}")
            ostia_left = None
            ostia_right = None
            label_artery = None
            ostia_results = None
            
        del vesselness_ostios  # Liberar memória
        del detected_circles # Liberar memória

        # 6. Segmentar artérias
        print("6. Segmentando artérias a partir dos óstios...")
        artery_results = segment_arteries_from_ostia(
            img_id, lcc_image, label_artery, ostia_left, ostia_right,
            run_config, artery_cache_dir
        )

        print(f"Pipeline completado com sucesso para a imagem {img_id}!\n")

        return {
            'image_id': img_id,
            'label_artery': label_artery,
            'aorta_mask': aorta_mask,
            'ostia_left': ostia_left,
            'ostia_right': ostia_right,
            'artery_results': artery_results,
            'ostia_results': ostia_results,
            'scaled_spacing': scaled_spacing
        }

    except Exception as e:
        print(f"Erro ao processar IMG {img_id}: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

print("Funcao de pipeline definida")

Funcao de pipeline definida


## Visualizar Óstios + Aorta + Label

In [4]:
pipeline_results = {}

for idx, row in selected_cases.iterrows():
    img_id = int(row['image_id'])
    resolution = row['resolution']
    bad_case_status = row['bad_case_status']

    if resolution == 'high':
        CONFIG["DOWNSCALE_FACTORS"] = [1, 1, 1]
    else:
        CONFIG["DOWNSCALE_FACTORS"] = [2, 2, 1]

    # Escalar config para a resolução base da configuração
    scaled_config = scale_config_to_resolution(CONFIG)

    # Executar pipeline
    result = run_pipeline_for_case(img_id, scaled_config)
    pipeline_results[img_id] = result

    print(f"\nIMG {img_id} ({resolution} res) - Status: {bad_case_status}")

    # Informações dos óstios
    ostia_left = result['ostia_left']
    ostia_right = result['ostia_right']

    print(f"Óstios detectados - Esquerdo: {len(ostia_left)}, Direito: {len(ostia_right)}")

    # Salvar aorta com óstios em HTML
    aorta_mask = result['aorta_mask']
    label = result['label']
    spacing = result['scaled_spacing']

    # Gerar visualização 3D da aorta com os óstios detectados
    try:
        visualize_aorta_with_ostia(
            aorta_mask,
            ostia_left,
            ostia_right,
            spacing=spacing,
            label_mask=label,
            use_physical_coords=True,
            save_html_path=HTML_OUTPUT_DIR / f"img_{img_id}_{resolution}_aorta_ostios.html",
            display_plot=False,
        )
    except Exception as e:
        print(f"   Erro ao gerar HTML: {e}")

    # Gerar visualização comparativa das artérias segmentadas vs label
    try:
        artery_mask = result['artery_results']['artery_mask']
        visualize_arteries_comparison(
            label_mask=label,
            predicted_mask=artery_mask,
            spacing=spacing,
            save_html_path=HTML_OUTPUT_DIR / f"img_{img_id}_{resolution}_arteries_comparison.html",
            display_plot=False,
        )
    except Exception as e:
        print(f"   Erro ao gerar HTML: {e}")

print("\nVisualizações de Detecção de Óstios + Aorta concluídas")


Processando IMG 431
1. Carregando e pré-processando a imagem...
2. Detectando óstios (círculos na aorta)...
Parada na fatia 192: Δr=3.00 ou dist=44.64
3. Segmentando aorta...
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/431_mask_aorta.npy
4. Calculando vesselness para ostios...
5. Detectando e avaliando óstios...
Erro ao processar IMG 431: Segundo óstio não encontrado! Tentou 2000 candidatos. Restrições: dist>=115.4, z_diff<=40.0mm, x_sep>=46.2. Tente aumentar 'top_n' ou relaxar os fatores.

IMG 431 (high res) - Status: error


Traceback (most recent call last):
  File "/tmp/ipykernel_203859/1747439437.py", line 56, in run_pipeline_for_case
    ostia_results = detect_and_evaluate_ostia(
        aorta_mask, vesselness_ostios, label, scaled_spacing, run_config
    )
  File "/home/matheus/workspace/coronary-centerline-detection/src/utils/segmentation/pipeline_steps.py", line 196, in detect_and_evaluate_ostia
    ostia_left, ostia_right = find_ostia(
                              ~~~~~~~~~~^
        aorta_mask,
        ^^^^^^^^^^^
    ...<7 lines>...
        erosion_radius=ostia_config["erosion_radius"],
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/matheus/workspace/coronary-centerline-detection/src/utils/segmentation/ostia_detection.py", line 223, in find_ostia
    raise ValueError(
    ...<4 lines>...
    )
ValueError: Segundo óstio não encontrado! Tentou 2000 candidatos. Restrições: dist>=115.4, z_diff<=40.0mm, x_sep>=46.2. Tente aumentar 'top_n' ou relaxar os fatores.


KeyError: 'ostia_left'